## Claim Extractor

In [21]:
"""
CASM — Claim Extractor (Hybrid Ensemble)
=========================================
Stage 1 of the CASM (Clinical Automated Safety Monitor) pipeline.

Single responsibility: convert a raw LLM clinical response into a list of
structured, typed, and entity-tagged atomic claims.

Dual-pipeline architecture:
  - Med7 (en_core_med7_trf)  → DRUG, DOSAGE, STRENGTH, FORM, FREQUENCY, ROUTE, DURATION
  - scispaCy (en_core_sci_md) → DISEASE, fallback conditions

Patient context and population flags are handled downstream by the
Knowledge Verifier — not here.

Install:
    pip install spacy scispacy torch
    pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_md-0.5.4.tar.gz
    pip install https://huggingface.co/kormilitzin/en_core_med7_trf/resolve/main/en_core_med7_trf-3.4.2.1-py3-none-any.whl

NumPy compatibility note:
    spaCy/thinc require NumPy < 2.0. If you hit a binary incompatibility error:
        pip install "numpy==1.26" --reinstall
        pip install spacy --reinstall --no-cache
"""

import re
import spacy
import torch
from dataclasses import dataclass
from enum import Enum
from typing import Optional


# ─── Med7 clinical labels ─────────────────────────────────────────────────────

# Entity labels recognised by Med7 as drug/medication names.
MED7_DRUG_LABELS    = {"DRUG"}

# Entity labels recognised by Med7 as dose or strength values (e.g. "200mg", "0.1%").
MED7_DOSAGE_LABELS  = {"DOSAGE", "STRENGTH"}

# Real clinical information from Med7 that is intentionally excluded from
# Claim.entities to keep the entity list focused on named clinical objects.
MED7_CLINICAL_NOISE = {"FREQUENCY", "DURATION", "FORM", "ROUTE"}


# ─── Enums ────────────────────────────────────────────────────────────────────

class ClaimType(str, Enum):
    """
    Classifies the medical nature of an extracted claim.

    Values
    ------
    DOSAGE_CLAIM      : Claim about drug dose, frequency, or administration route.
    DRUG_SAFETY_CLAIM : Claim about drug hazards, contraindications, or warnings.
    DRUG_INTERACTION  : Claim about interactions between two or more drugs.
    PROCEDURAL_CLAIM  : Claim about monitoring, lab tests, or follow-up actions.
    DIAGNOSIS_CLAIM   : Claim about a patient diagnosis or clinical presentation.
    POPULATION_CLAIM  : Claim specific to a patient sub-population (e.g. elderly,
                        pregnant, renal-impaired).
    GENERAL_MEDICAL   : Fallback for claims with medical content that does not
                        fit a more specific category.
    """
    DOSAGE_CLAIM      = "DOSAGE_CLAIM"
    DRUG_SAFETY_CLAIM = "DRUG_SAFETY_CLAIM"
    DRUG_INTERACTION  = "DRUG_INTERACTION"
    PROCEDURAL_CLAIM  = "PROCEDURAL_CLAIM"
    DIAGNOSIS_CLAIM   = "DIAGNOSIS_CLAIM"
    POPULATION_CLAIM  = "POPULATION_CLAIM"
    GENERAL_MEDICAL   = "GENERAL_MEDICAL"


# ─── Data Classes ─────────────────────────────────────────────────────────────

@dataclass
class Entity:
    """
    A single named entity recognised within a claim sentence.

    Attributes
    ----------
    text  : The surface form of the entity as it appears in the sentence.
    label : The NER label assigned by the pipeline that detected it.
              Med7 labels  : DRUG | DOSAGE | STRENGTH | FORM | FREQUENCY | ROUTE | DURATION
              scispaCy labels: DISEASE | CONDITION | DISORDER
    start : Character offset (inclusive) of the entity within the sentence.
    end   : Character offset (exclusive) of the entity within the sentence.
    """
    text:  str
    label: str
    start: int
    end:   int


@dataclass
class Claim:
    """
    A single structured, atomic medical claim extracted from an LLM response.

    Attributes
    ----------
    claim_id              : Unique identifier for this claim (e.g. ``"claim_0"``).
    claim_text            : The raw sentence text from which the claim was derived.
    claim_type            : Semantic category of the claim (see :class:`ClaimType`).
    entities              : All named entities detected by the hybrid pipeline.
    drug_names            : Deduplicated list of drug/medication names found in the claim.
    dosages               : Deduplicated list of dose/strength values (e.g. ``"200mg"``).
    conditions            : Deduplicated list of diseases or clinical conditions mentioned.
    requires_verification : ``True`` when the claim type demands downstream fact-checking
                            (DOSAGE_CLAIM, DRUG_SAFETY_CLAIM, DRUG_INTERACTION, or
                            POPULATION_CLAIM).
    confidence            : Placeholder score in [0, 1]; populated by the Confidence
                            Calibrator stage — always ``0.0`` at extraction time.
    """
    claim_id:              str
    claim_text:            str
    claim_type:            ClaimType
    entities:              list[Entity]
    drug_names:            list[str]
    dosages:               list[str]
    conditions:            list[str]
    requires_verification: bool
    confidence:            float


# ─── Keyword Maps ─────────────────────────────────────────────────────────────

# Maps each ClaimType to a list of trigger keywords used for rule-based
# classification.  The type whose keywords score the most matches in a
# sentence wins; ties default to GENERAL_MEDICAL.
CLAIM_TYPE_KEYWORDS: dict[ClaimType, list[str]] = {
    ClaimType.DOSAGE_CLAIM: [
        "mg", "mcg", "ml", "%", "dose", "dosage", "twice", "once", "three times",
        "daily", "bid", "tid", "qid", "weekly", "monthly", "prescribe",
        "administer", "give", "start", "initiate", "tablet", "capsule", "drops",
    ],
    ClaimType.DRUG_SAFETY_CLAIM: [
        "avoid", "contraindicated", "do not use", "caution", "warning",
        "unsafe", "dangerous", "harmful", "risk", "adverse", "side effect",
        "toxicity", "nephrotoxic", "hepatotoxic", "cardiotoxic",
    ],
    ClaimType.DRUG_INTERACTION: [
        "interaction", "interacts", "combined with", "together with",
        "concurrent", "concomitant", "potentiates", "inhibits", "induces",
        "bleeding risk", "increased risk when",
    ],
    ClaimType.PROCEDURAL_CLAIM: [
        "monitor", "check", "test", "measure", "assess", "every", "weeks",
        "months", "follow up", "review", "screen", "scan", "blood test",
        "lab", "hba1c", "creatinine", "egfr", "blood pressure",
    ],
    ClaimType.DIAGNOSIS_CLAIM: [
        "diagnosis", "diagnose", "patient has", "presents with", "suffering from",
        "consistent with", "indicative of", "suggests", "confirms",
    ],
    ClaimType.POPULATION_CLAIM: [
        "elderly", "pediatric", "children", "pregnant", "renal", "kidney",
        "hepatic", "liver", "geriatric", "neonatal", "trimester",
    ],
}

# Regex patterns for conjunctive phrases that often join two independent
# clinical instructions within a single sentence.  Matching these allows
# compound sentences to be split into separate atomic claims.
SPLIT_CONJUNCTIONS = [
    r"\band also\b", r"\badditionally\b", r"\bfurthermore\b",
    r"\bin addition\b", r"\bmoreover\b", r"\bhowever\b", r"\balternatively\b",
]


# ─── Extractor ────────────────────────────────────────────────────────────────

class ClaimExtractor:
    """
    Hybrid Ensemble Claim Extractor.

    Converts a raw LLM clinical response into a list of structured
    :class:`Claim` objects by running two complementary NLP pipelines:

    Pipeline 1 — Med7 (``en_core_med7_trf``):
        Handles DRUG, DOSAGE, STRENGTH, FORM, FREQUENCY, ROUTE, DURATION.
        High precision for clinical instructions.
        Transformer-based — benefits significantly from GPU acceleration.

    Pipeline 2 — scispaCy (``en_core_sci_md``):
        Fallback for DISEASE/CONDITION detection.
        Only disease-like entities are kept; generic ``ENTITY`` noise is discarded.

    GPU:
        If a CUDA-capable GPU is available, spaCy will automatically use it
        for the Med7 transformer pipeline. ``spacy.prefer_gpu()`` is called
        before any model is loaded (handled in :meth:`__init__`).

    Example usage::

        extractor = ClaimExtractor()
        claims = extractor.extract_as_dict("Prescribe Metformin 500mg twice daily.")
    """

    def __init__(
        self,
        med7_model:   str  = "en_core_med7_trf",
        sci_model:    str  = "en_core_sci_md",
        require_gpu:  bool = False,
    ):
        """
        Initialise the hybrid extractor and load both NLP models.

        Args:
            med7_model  : spaCy model name for the Med7 transformer pipeline.
                          Defaults to ``"en_core_med7_trf"``.
            sci_model   : spaCy model name for the scispaCy pipeline.
                          Defaults to ``"en_core_sci_md"``.
            require_gpu : If ``True``, raises :exc:`RuntimeError` when no
                          CUDA GPU is detected.  Useful in production to
                          prevent silent CPU fallback.  Defaults to ``False``.
        """
        self._setup_device(require_gpu=require_gpu)

        print(f"[ClaimExtractor] Loading Med7 model    : {med7_model}")
        self.nlp_med7 = spacy.load(med7_model)

        print(f"[ClaimExtractor] Loading scispaCy model: {sci_model}")
        self.nlp_sci  = spacy.load(sci_model)

        print("[ClaimExtractor] Hybrid ensemble ready.")

    # ── Device Setup ──────────────────────────────────────────────────────────

    @staticmethod
    def _setup_device(require_gpu: bool = False) -> None:
        """
        Configure spaCy's compute device before any model is loaded.

        Must be called BEFORE ``spacy.load()`` — spaCy allocates model tensors
        at load time and will not migrate them afterwards.

        Priority:
            1. CUDA GPU  (via PyTorch + ``spacy.prefer_gpu()``)
            2. CPU       (silent fallback unless ``require_gpu=True``)

        Note on spaCy GPU=OFF:
            Med7 (``en_core_med7_trf`` 3.4.x) was compiled against spaCy 3.4.
            Running it under spaCy 3.7 means the transformer backend may not
            activate GPU even when CUDA is available — this is a model/version
            mismatch, not a code error. The model still runs correctly on CPU.
            To get full GPU utilisation, retrain Med7 against spaCy 3.7 or
            wait for an official 3.7-compatible release.

        Args:
            require_gpu : If ``True`` and no CUDA device is found, raises
                          :exc:`RuntimeError`.
        """
        if torch.cuda.is_available():
            activated = spacy.prefer_gpu()
            gpu_name  = torch.cuda.get_device_name(0)
            vram_gb   = torch.cuda.get_device_properties(0).total_memory / 1e9
            status    = "ON" if activated else "OFF (model version mismatch — see docstring)"
            print(
                f"[ClaimExtractor] GPU detected : {gpu_name} "
                f"({vram_gb:.1f} GB VRAM) — spaCy GPU={status}"
            )
        else:
            if require_gpu:
                raise RuntimeError(
                    "[ClaimExtractor] require_gpu=True but no CUDA GPU found. "
                    "Install CUDA + torch GPU build, or set require_gpu=False."
                )
            print("[ClaimExtractor] No GPU detected — running on CPU.")

    @staticmethod
    def device_info() -> dict:
        """
        Return a summary of the current compute environment.

        Returns
        -------
        dict
            Keys: ``cuda_available``, ``cuda_device_count``, ``cuda_device_name``,
            ``cuda_vram_gb``, ``spacy_gpu_active``, ``torch_version``,
            ``spacy_version``.
        """
        return {
            "cuda_available":    torch.cuda.is_available(),
            "cuda_device_count": torch.cuda.device_count(),
            "cuda_device_name":  torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
            "cuda_vram_gb":      round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2)
                                 if torch.cuda.is_available() else None,
            "spacy_gpu_active":  spacy.prefer_gpu() if torch.cuda.is_available() else False,
            "torch_version":     torch.__version__,
            "spacy_version":     spacy.__version__,
        }

    # ── Public API ────────────────────────────────────────────────────────────

    def extract(self, llm_response: str) -> list[Claim]:
        """
        Extract a list of :class:`Claim` objects from raw LLM output text.

        The text is first split into atomic sentences, then each sentence is
        processed independently through the hybrid NLP pipeline.  Sentences
        with no detectable medical content are silently dropped.

        Args:
            llm_response : Raw text produced by the LLM.

        Returns:
            A list of :class:`Claim` objects, one per extracted sentence.
        """
        sentences = self._split_into_sentences(llm_response)
        claims = []
        for idx, sentence in enumerate(sentences):
            if not sentence.strip():
                continue
            claim = self._process_sentence(sentence, idx)
            if claim:
                claims.append(claim)
        print(f"[ClaimExtractor] {len(claims)} claim(s) extracted.")
        return claims

    def extract_as_dict(self, llm_response: str) -> list[dict]:
        """
        Extract claims and return them as a list of JSON-serialisable dicts.

        Convenience wrapper around :meth:`extract` — delegates all logic there
        and converts the result via :meth:`_to_dict`.

        Args:
            llm_response : Raw text produced by the LLM.

        Returns:
            A list of plain ``dict`` objects mirroring the :class:`Claim` schema.
        """
        return [self._to_dict(c) for c in self.extract(llm_response)]

    # ── Sentence Splitting ────────────────────────────────────────────────────

    def _split_into_sentences(self, text: str) -> list[str]:
        """
        Split raw text into a flat list of atomic sentence strings.

        Two-stage process:
          1. scispaCy's sentence boundary detector splits on full stops and
             other hard boundaries.
          2. :meth:`_split_on_conjunctions` further splits compound sentences
             that join two independent clinical instructions with a conjunction.

        Sentences shorter than four words are discarded as fragments.

        Args:
            text : Raw input text (may contain multiple sentences).

        Returns:
            List of cleaned, atomic sentence strings.
        """
        # Use scispaCy for sentence boundary detection
        doc = self.nlp_sci(text)
        sentences = [sent.text.strip() for sent in doc.sents]
        final = []
        for sentence in sentences:
            final.extend(self._split_on_conjunctions(sentence))
        return [s for s in final if len(s.split()) >= 4]

    def _split_on_conjunctions(self, sentence: str) -> list[str]:
        """
        Split a single sentence on clinical conjunctions from
        :data:`SPLIT_CONJUNCTIONS`.

        Each pattern in the list is applied iteratively so that sentences
        containing multiple conjunctions are split into the correct number of
        parts.

        Args:
            sentence : A single sentence string.

        Returns:
            A list of one or more sub-sentence strings.
        """
        parts = [sentence]
        for pattern in SPLIT_CONJUNCTIONS:
            new_parts = []
            for part in parts:
                split = re.split(pattern, part, flags=re.IGNORECASE)
                new_parts.extend([s.strip() for s in split if s.strip()])
            parts = new_parts
        return parts

    # ── Claim Processing ──────────────────────────────────────────────────────

    def _process_sentence(self, sentence: str, idx: int) -> Optional[Claim]:
        """
        Run the hybrid NLP pipeline on a single sentence and return a
        :class:`Claim`, or ``None`` if the sentence contains no medical content.

        Steps:
          1. Med7 identifies drugs, dosages, and other clinical structure.
          2. scispaCy identifies diseases/conditions not already covered by Med7.
          3. Both entity lists are merged (Med7 is authoritative; scispaCy fills gaps).
          4. Derived fields (drug_names, dosages, conditions) are extracted.
          5. The claim type is classified via keyword scoring.

        Args:
            sentence : A single atomic sentence string.
            idx      : Zero-based position of this sentence in the original text,
                       used to generate a stable ``claim_id``.

        Returns:
            A populated :class:`Claim` object, or ``None`` if the sentence has
            no recognisable medical content.
        """
        # ── Pipeline 1: Med7 — drugs, dosages, clinical structure ─────────────
        doc_med7 = self.nlp_med7(sentence)
        med7_entities = [
            Entity(
                text=ent.text,
                label=ent.label_,
                start=ent.start_char,
                end=ent.end_char,
            )
            for ent in doc_med7.ents
        ]

        # ── Pipeline 2: scispaCy — diseases & conditions only ─────────────────
        doc_sci = self.nlp_sci(sentence)
        DISEASE_LABELS = {"DISEASE", "CONDITION", "DISORDER"}
        sci_disease_entities = [
            Entity(
                text=ent.text,
                label=ent.label_,
                start=ent.start_char,
                end=ent.end_char,
            )
            for ent in doc_sci.ents
            if ent.label_ in DISEASE_LABELS
        ]

        # ── Merge: Med7 authoritative; scispaCy fills disease gaps ────────────
        med7_spans    = {(e.start, e.end) for e in med7_entities}
        merged_entities = list(med7_entities)
        for e in sci_disease_entities:
            if (e.start, e.end) not in med7_spans:
                merged_entities.append(e)

        drug_names = self._extract_drugs(med7_entities, sentence)
        dosages    = self._extract_dosages(med7_entities, sentence)
        conditions = self._extract_conditions(sci_disease_entities, sentence)
        claim_type = self._classify_claim_type(sentence, merged_entities)

        # Drop sentences with no medical content at all
        if claim_type == ClaimType.GENERAL_MEDICAL and not merged_entities:
            return None

        requires_verification = claim_type in {
            ClaimType.DOSAGE_CLAIM,
            ClaimType.DRUG_SAFETY_CLAIM,
            ClaimType.DRUG_INTERACTION,
            ClaimType.POPULATION_CLAIM,
        }

        return Claim(
            claim_id=f"claim_{idx}",
            claim_text=sentence,
            claim_type=claim_type,
            entities=merged_entities,
            drug_names=drug_names,
            dosages=dosages,
            conditions=conditions,
            requires_verification=requires_verification,
            confidence=0.0,
        )

    # ── Claim Type Classification ─────────────────────────────────────────────

    def _classify_claim_type(
        self, sentence: str, entities: list[Entity]
    ) -> ClaimType:
        """
        Assign a :class:`ClaimType` to a sentence using keyword frequency scoring.

        Each :data:`CLAIM_TYPE_KEYWORDS` entry is checked against the
        lower-cased sentence.  The type with the most keyword matches wins.
        If no keywords match at all, :attr:`ClaimType.GENERAL_MEDICAL` is returned.

        Args:
            sentence : The raw sentence text.
            entities : Merged entity list (currently unused — reserved for a
                       future entity-boosted scoring pass).

        Returns:
            The best-matching :class:`ClaimType`.
        """
        sentence_lower = sentence.lower()
        scores: dict[ClaimType, int] = {ct: 0 for ct in ClaimType}

        for claim_type, keywords in CLAIM_TYPE_KEYWORDS.items():
            for keyword in keywords:
                if keyword in sentence_lower:
                    scores[claim_type] += 1

        best = max(scores, key=lambda ct: scores[ct])
        return best if scores[best] > 0 else ClaimType.GENERAL_MEDICAL

    # ── Entity Extraction ─────────────────────────────────────────────────────

    @staticmethod
    def _clean_drug_name(text: str) -> str:
        """
        Normalise a raw Med7 drug span by removing common extraction artefacts.

        Artefacts handled:
          - Parenthetical abbreviations appended by the LLM, e.g.
            ``"Fluorometholone (FML)"`` → ``"Fluorometholone"``
          - Leading or trailing punctuation and bracket characters.

        Args:
            text : Raw drug span text as returned by Med7.

        Returns:
            Cleaned drug name string, or an empty string if nothing remains
            after cleaning (caller should discard empty results).
        """
        # Remove parenthetical suffix:  'Drug (ABBR)' → 'Drug'
        text = re.sub(r'\s*\(.*?\)\s*$', '', text)
        # Strip remaining leading/trailing non-alphanumeric characters
        text = re.sub(r'^[^A-Za-z]+|[^A-Za-z0-9]+$', '', text)
        return text.strip()

    def _extract_drugs(
        self, med7_entities: list[Entity], sentence: str
    ) -> list[str]:
        """
        Build a deduplicated list of drug names from Med7 entities and a
        regex suffix fallback.

        Primary source: Med7 ``DRUG`` entities — high-precision clinical NER.
        Fallback: regex matching capitalised tokens whose suffixes are common
        in drug nomenclature (e.g. ``-olone``, ``-mycin``, ``-statin``).

        Args:
            med7_entities : Entities extracted by the Med7 pipeline.
            sentence      : The original sentence text (used for regex fallback).

        Returns:
            Ordered, deduplicated list of drug name strings.
        """
        # Primary: Med7 DRUG label — high precision
        drugs = [
            self._clean_drug_name(e.text)
            for e in med7_entities
            if e.label in MED7_DRUG_LABELS
        ]
        drugs = [d for d in drugs if d]  # drop empty strings after cleaning

        # Secondary: regex suffix fallback (catches brand names Med7 misses)
        regex = re.findall(
            r'\b[A-Z][a-z]+(?:in|ol|ine|ide|ate|mab|nib|pril|sartan|statin|mycin|cillin|zole|olone)\b',
            sentence,
        )
        for d in regex:
            if d not in drugs:
                drugs.append(d)

        return list(dict.fromkeys(drugs))  # deduplicate, preserve order

    def _extract_dosages(
        self, med7_entities: list[Entity], sentence: str
    ) -> list[str]:
        """
        Build a deduplicated list of dose/strength values from Med7 entities
        and a regex numeric fallback.

        Primary source: Med7 ``DOSAGE`` and ``STRENGTH`` entities.
        Fallback: regex matching numeric values followed by a recognised unit
        symbol (``mg``, ``mcg``, ``ml``, ``g``, ``%``, etc.).

        Args:
            med7_entities : Entities extracted by the Med7 pipeline.
            sentence      : The original sentence text (used for regex fallback).

        Returns:
            Ordered, deduplicated list of dosage strings.
        """
        # Primary: Med7 DOSAGE / STRENGTH labels
        dosages = [e.text for e in med7_entities if e.label in MED7_DOSAGE_LABELS]

        # Secondary: regex — catches formats Med7 might miss
        regex = re.findall(
            r'\b\d+(?:\.\d+)?\s*(?:mg|mcg|ml|g|%|units?|IU|mmol)(?:/(?:day|kg|dose))?\b',
            sentence,
            re.IGNORECASE,
        )
        for d in regex:
            if d.strip() not in dosages:
                dosages.append(d.strip())

        return list(dict.fromkeys(dosages))

    def _extract_conditions(
        self, sci_entities: list[Entity], sentence: str
    ) -> list[str]:
        """
        Build a deduplicated list of disease/condition names from scispaCy
        entities and a known-abbreviation fallback.

        Primary source: scispaCy entities whose label is ``DISEASE``,
        ``CONDITION``, or ``DISORDER`` — the generic ``ENTITY`` label is
        explicitly excluded to suppress noise.

        Fallback: regex matching a fixed vocabulary of common clinical
        abbreviations (e.g. ``CKD``, ``T2DM``, ``UTI``).

        Args:
            sci_entities : Disease-filtered entities from the scispaCy pipeline.
            sentence     : The original sentence text (used for abbreviation
                           fallback).

        Returns:
            Ordered, deduplicated list of condition/disease strings.
        """
        # Primary: scispaCy disease labels only (no ENTITY noise)
        conditions = [e.text for e in sci_entities]

        # Secondary: known clinical abbreviations
        abbrevs = re.findall(
            r'\b(?:CKD|DM|T2DM|HTN|CAD|CHF|COPD|AF|DVT|PE|UTI|PRK|TransPRK)\b',
            sentence,
        )
        for a in abbrevs:
            if a not in conditions:
                conditions.append(a)

        return list(dict.fromkeys(conditions))

    # ── Serialisation ─────────────────────────────────────────────────────────

    def _to_dict(self, claim: Claim) -> dict:
        """
        Serialise a :class:`Claim` to a plain, JSON-serialisable dictionary.

        The ``claim_type`` enum is converted to its string value so the result
        can be passed directly to ``json.dumps`` or a DataFrame constructor.

        Args:
            claim : The :class:`Claim` instance to serialise.

        Returns:
            A ``dict`` with the same keys as the :class:`Claim` dataclass.
        """
        return {
            "claim_id":              claim.claim_id,
            "claim_text":            claim.claim_text,
            "claim_type":            claim.claim_type.value,
            "entities":              [
                {"text": e.text, "label": e.label} for e in claim.entities
            ],
            "drug_names":            claim.drug_names,
            "dosages":               claim.dosages,
            "conditions":            claim.conditions,
            "requires_verification": claim.requires_verification,
            "confidence":            claim.confidence,
        }


## Claim Extractor Test

In [22]:
testdict = ClaimExtractor()

[ClaimExtractor] No GPU detected — running on CPU.
[ClaimExtractor] Loading Med7 model    : en_core_med7_trf


C:\Users\basim\PycharmProjects\Clinical-AI-Safety-Monitor\.venv\Lib\site-packages\spacy\util.py:910: UserWarning: [W095] Model 'en_core_med7_trf' (3.4.2.1) was trained with spaCy v3.4.2 and may not be 100% compatible with the current version (3.7.4). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)
C:\Users\basim\PycharmProjects\Clinical-AI-Safety-Monitor\.venv\Lib\site-packages\spacy_transformers\layers\hf_shim.py:137: UserWarning: Error loading saved torch state_dict with strict=True, likely due to differences between 'transformers' versions. Attempting to load with strict=False as a fallback...

If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current 'transformers' and 'spacy-transformers' versions. For more details and available updat

[ClaimExtractor] Loading scispaCy model: en_core_sci_md
[ClaimExtractor] Hybrid ensemble ready.


In [23]:
test1 = testdict.extract_as_dict(llm_response="To treat corneal haze 5 months post-TransPRK, prescribe Fluorometholone (FML) 0.1% eye drops four times daily for 4 weeks. You may also consider Oral Vitamin C (1000mg/day) to support corneal healing.")

[ClaimExtractor] 2 claim(s) extracted.


In [24]:
test1

[{'claim_id': 'claim_0',
  'claim_text': 'To treat corneal haze 5 months post-TransPRK, prescribe Fluorometholone (FML) 0.1% eye drops four times daily for 4 weeks.',
  'claim_type': 'DOSAGE_CLAIM',
  'entities': [{'text': 'Fluorometholone (', 'label': 'DRUG'},
   {'text': '0.1%', 'label': 'STRENGTH'},
   {'text': 'eye', 'label': 'ROUTE'},
   {'text': 'drops', 'label': 'FORM'},
   {'text': 'four times daily', 'label': 'FREQUENCY'},
   {'text': 'for 4 weeks', 'label': 'DURATION'}],
  'drug_names': ['Fluorometholone'],
  'dosages': ['0.1%'],
  'conditions': ['TransPRK'],
  'requires_verification': True,
  'confidence': 0.0},
 {'claim_id': 'claim_1',
  'claim_text': 'You may also consider Oral Vitamin C (1000mg/day) to support corneal healing.',
  'claim_type': 'DOSAGE_CLAIM',
  'entities': [{'text': 'Oral', 'label': 'ROUTE'},
   {'text': 'Vitamin C', 'label': 'DRUG'},
   {'text': '1000mg/day', 'label': 'STRENGTH'}],
  'drug_names': ['Vitamin C', 'Vitamin'],
  'dosages': ['1000mg/day'],
 

In [25]:
test2 = testdict.extract_as_dict(llm_response="For a patient with a confirmed UTI, prescribe Trimethoprim 200mg twice daily for 7 days. If symptoms persist after 48 hours, switch to Nitrofurantoin 100mg four times daily for 5 days. Monitor renal function with creatinine levels weekly.")

[ClaimExtractor] 3 claim(s) extracted.


In [26]:
test2

[{'claim_id': 'claim_0',
  'claim_text': 'For a patient with a confirmed UTI, prescribe Trimethoprim 200mg twice daily for 7 days.',
  'claim_type': 'DOSAGE_CLAIM',
  'entities': [{'text': 'Trimethoprim', 'label': 'DRUG'},
   {'text': '200mg', 'label': 'STRENGTH'},
   {'text': 'twice daily', 'label': 'FREQUENCY'},
   {'text': 'for 7 days', 'label': 'DURATION'}],
  'drug_names': ['Trimethoprim'],
  'dosages': ['200mg'],
  'conditions': ['UTI'],
  'requires_verification': True,
  'confidence': 0.0},
 {'claim_id': 'claim_1',
  'claim_text': 'If symptoms persist after 48 hours, switch to Nitrofurantoin 100mg four times daily for 5 days.',
  'claim_type': 'DOSAGE_CLAIM',
  'entities': [{'text': 'Nitrofurantoin', 'label': 'DRUG'},
   {'text': '100mg', 'label': 'STRENGTH'},
   {'text': 'four times daily', 'label': 'FREQUENCY'},
   {'text': 'for 5 days', 'label': 'DURATION'}],
  'drug_names': ['Nitrofurantoin'],
  'dosages': ['100mg'],
  'conditions': [],
  'requires_verification': True,
  'co

In [27]:
test3 = testdict.extract_as_dict(llm_response="In a T2DM patient with CKD stage 3, avoid Metformin 500mg if eGFR drops below 30 ml/min due to risk of lactic acidosis. Consider switching to Sitagliptin 50mg once daily, which is renally dosed and safer in this population. Monitor HbA1c every 3 months.")

[ClaimExtractor] 3 claim(s) extracted.


In [28]:
test3

[{'claim_id': 'claim_0',
  'claim_text': 'In a T2DM patient with CKD stage 3, avoid Metformin 500mg if eGFR drops below 30 ml/min due to risk of lactic acidosis.',
  'claim_type': 'DOSAGE_CLAIM',
  'entities': [{'text': 'Metformin', 'label': 'DRUG'},
   {'text': '500mg', 'label': 'STRENGTH'}],
  'drug_names': ['Metformin'],
  'dosages': ['500mg', '30 ml'],
  'conditions': ['T2DM', 'CKD'],
  'requires_verification': True,
  'confidence': 0.0},
 {'claim_id': 'claim_1',
  'claim_text': 'Consider switching to Sitagliptin 50mg once daily, which is renally dosed and safer in this population.',
  'claim_type': 'DOSAGE_CLAIM',
  'entities': [{'text': 'Sitagliptin', 'label': 'DRUG'},
   {'text': '50mg', 'label': 'STRENGTH'},
   {'text': 'once daily', 'label': 'FREQUENCY'}],
  'drug_names': ['Sitagliptin'],
  'dosages': ['50mg'],
  'conditions': [],
  'requires_verification': True,
  'confidence': 0.0},
 {'claim_id': 'claim_2',
  'claim_text': 'Monitor HbA1c every 3 months.',
  'claim_type': 'PR

## Knowledge Verifier

In [29]:
"""
CASM — Knowledge Verifier
=========================
Stage 2 of the CASM pipeline.

Single responsibility: verify each Claim that has requires_verification=True
and produce an EvidenceResult containing an NLI verdict, confidence score,
supporting PubMed abstracts, and openFDA adverse-event counts.

Architecture
------------
For each verifiable claim:
  1. Encode claim text with S-PubMedBERT (768-d vector)
  2. Semantic search → Chroma (pubmed_abstracts collection, top-K)
  3. If Chroma returns < MIN_HITS, fetch live abstracts from PubMed Entrez
     and upsert them so the collection grows over time.
  4. NLI (BiomedNLP-PubMedBERT MNLI) on each abstract → SUPPORTED /
     CONTRADICTED / AMBIGUOUS + confidence
  5. Aggregate 5 NLI verdicts: majority wins; CONTRADICTED beats AMBIGUOUS
     in a tie.
  6. openFDA /drug/event.json → adverse-event count (parallel, async-style
     via a thread pool so NLI and FDA query overlap).
  7. Produce EvidenceResult.

Population flags are intentionally excluded from this stage.
"""

import re
import time
import concurrent.futures
import hashlib
from dataclasses import dataclass, field
from enum import Enum
from typing import Optional
import requests
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings


In [30]:
# ── Enums & Data Classes ───────────────────────────────────────────────────────

class NLIVerdict(str, Enum):
    """Outcome of the NLI aggregation step for a single claim."""
    SUPPORTED     = "SUPPORTED"
    CONTRADICTED  = "CONTRADICTED"
    AMBIGUOUS     = "AMBIGUOUS"


@dataclass
class SearchResult:
    """
    A single PubMed abstract retrieved from the Chroma vector store.

    Attributes
    ----------
    pmid        : PubMed identifier string (e.g. "38123456").
    abstract    : Full abstract text used as NLI premise.
    title       : Article title for display / logging.
    distance    : Cosine distance from the query vector (lower = more similar).
    drug_filter : Drug name used to narrow this fetch, if any.
    """
    pmid:        str
    abstract:    str
    title:       str   = ""
    distance:    float = 0.0
    drug_filter: str   = ""


@dataclass
class EvidenceResult:
    """
    Structured verification output for a single Claim.

    Attributes
    ----------
    claim_id       : Mirrors Claim.claim_id.
    claim_text     : Original claim text (for display).
    verdict        : Aggregated NLI verdict across all retrieved abstracts.
    confidence     : Weighted mean confidence of the majority-verdict abstracts.
    pubmed_hits    : Top-K SearchResult objects used for NLI.
    fda_count      : Total adverse event reports for the primary drug on openFDA.
    fda_serious    : Subset of fda_count flagged as serious outcomes.
    nli_scores     : Per-abstract NLI confidence scores (same order as pubmed_hits).
    nli_verdicts   : Per-abstract NLI verdict labels.
    skipped        : True when requires_verification was False (claim not checked).
    error          : Non-empty string when verification failed with an exception.
    """
    claim_id:     str
    claim_text:   str
    verdict:      NLIVerdict          = NLIVerdict.AMBIGUOUS
    confidence:   float               = 0.0
    pubmed_hits:  list[SearchResult]  = field(default_factory=list)
    fda_count:    int                 = 0
    fda_serious:  int                 = 0
    nli_scores:   list[float]         = field(default_factory=list)
    nli_verdicts: list[str]           = field(default_factory=list)
    skipped:      bool                = False
    error:        str                 = ""

    def to_dict(self) -> dict:
        """
        Serialise an :class:`EvidenceResult` to a JSON-safe dictionary.

        Rounds floating-point confidence values for stable display and keeps
        only lightweight metadata for PubMed hits (PMID, title, distance).

        Returns:
            A plain ``dict`` suitable for ``json.dumps`` or tabular export.
        """
        return {
            "claim_id":    self.claim_id,
            "claim_text":  self.claim_text,
            "verdict":     self.verdict.value,
            "confidence":  round(self.confidence, 4),
            "fda_count":   self.fda_count,
            "fda_serious": self.fda_serious,
            "nli_scores":  [round(s, 4) for s in self.nli_scores],
            "nli_verdicts": self.nli_verdicts,
            "pubmed_hits": [
                {"pmid": h.pmid, "title": h.title, "distance": round(h.distance, 4)}
                for h in self.pubmed_hits
            ],
            "skipped":     self.skipped,
            "error":       self.error,
        }

print("✅ EvidenceResult dataclass defined")

✅ EvidenceResult dataclass defined


In [31]:
# ── Chroma Vector Store ────────────────────────────────────────────────────────

CHROMA_PATH       = "./chroma_store"   # persisted on disk; change to a network path if needed
COLLECTION_NAME   = "pubmed_abstracts"
TOP_K             = 5                  # abstracts retrieved per claim
MIN_HITS          = 2                  # minimum Chroma hits before triggering live fetch
EMBED_MODEL_NAME  = "pritamdeka/S-PubMedBert-MS-MARCO"


class ChromaStore:
    """
    Thin wrapper around a persistent Chroma collection that stores
    PubMed abstracts as 768-d S-PubMedBERT embeddings.

    The embedding model is loaded once and reused for both indexing and querying.
    """

    def __init__(
        self,
        embed_model: str = EMBED_MODEL_NAME,
        chroma_path: str = CHROMA_PATH,
        collection:  str = COLLECTION_NAME,
    ):
        """
        Initialise the Chroma embedding store and open/create its collection.

        Args:
            embed_model : SentenceTransformer model used to encode abstracts
                          and claims into the shared embedding space.
            chroma_path : On-disk persistence path for the Chroma database.
            collection  : Name of the Chroma collection to open or create.
        """
        print(f"[ChromaStore] Loading embedding model: {embed_model}")
        self.embedder = SentenceTransformer(embed_model)

        print(f"[ChromaStore] Opening Chroma store at: {chroma_path}")
        self.client = chromadb.PersistentClient(path=chroma_path)
        self.col    = self.client.get_or_create_collection(
            name=collection,
            metadata={"hnsw:space": "cosine"},
        )
        print(f"[ChromaStore] Collection '{collection}' — {self.col.count()} docs")

    # ── Encoding ──────────────────────────────────────────────────────────────

    def encode(self, text: str) -> list[float]:
        """
        Encode free text into a normalised embedding vector.

        Args:
            text : Input text to embed (claim or abstract).

        Returns:
            A normalised dense vector as a Python list of floats.
        """
        return self.embedder.encode(text, normalize_embeddings=True).tolist()

    # ── Query ─────────────────────────────────────────────────────────────────

    def query(
        self,
        claim_text: str,
        top_k:      int          = TOP_K,
        drug_filter: Optional[str] = None,
    ) -> list[SearchResult]:
        """
        Run semantic search and return the closest PubMed abstracts.

        The claim text is embedded with the store's encoder and queried against
        the persisted Chroma collection using cosine distance.

        Args:
            claim_text  : Raw claim sentence to use as the semantic query.
            top_k       : Maximum number of nearest-neighbour hits to return.
            drug_filter : Optional drug name filter applied to collection
                          metadata (``drug`` field). Falls back to an
                          unfiltered query if the filtered query fails.

        Returns:
            A list of :class:`SearchResult` objects sorted by vector distance.
        """
        vector = self.encode(claim_text)

        where = {"drug": {"$eq": drug_filter.lower()}} if drug_filter else None

        try:
            results = self.col.query(
                query_embeddings=[vector],
                n_results=min(top_k, max(self.col.count(), 1)),
                where=where,
                include=["documents", "metadatas", "distances"],
            )
        except Exception:
            # Fall back without drug filter if filter fails (empty collection, etc.)
            results = self.col.query(
                query_embeddings=[vector],
                n_results=min(top_k, max(self.col.count(), 1)),
                include=["documents", "metadatas", "distances"],
            )

        hits: list[SearchResult] = []
        docs      = results.get("documents", [[]])[0]
        metas     = results.get("metadatas",  [[]])[0]
        distances = results.get("distances",  [[]])[0]

        for doc, meta, dist in zip(docs, metas, distances):
            hits.append(SearchResult(
                pmid        = meta.get("pmid", "unknown"),
                abstract    = doc,
                title       = meta.get("title", ""),
                distance    = float(dist),
                drug_filter = drug_filter or "",
            ))
        return hits

    # ── Upsert ────────────────────────────────────────────────────────────────

    def upsert(self, abstracts: list[dict]) -> int:
        """
        Add or update PubMed abstracts in the Chroma collection.

        Expected per-item keys:
            - ``pmid`` (required)
            - ``abstract`` (required)
            - ``title`` (optional)
            - ``drug`` (optional; normalised to lowercase)

        Args:
            abstracts : Batch of abstract dictionaries to index.

        Returns:
            Number of documents successfully upserted.
        """
        if not abstracts:
            return 0

        ids, docs, metas, embeds = [], [], [], []
        for item in abstracts:
            pmid = str(item["pmid"])
            text = item.get("abstract", "")
            if not text.strip():
                continue
            ids.append(pmid)
            docs.append(text)
            metas.append({
                "pmid":  pmid,
                "title": item.get("title", ""),
                "drug":  item.get("drug",  "").lower(),
            })
            embeds.append(self.encode(text))

        if ids:
            self.col.upsert(ids=ids, documents=docs, metadatas=metas, embeddings=embeds)
        return len(ids)


# Instantiate globally so both the live-fetch helper and the verifier share it
chroma = ChromaStore()
print(f"\n[ChromaStore] Ready — {chroma.col.count()} abstracts in store")

[2026-04-16 19:29:54,227] INFO — sentence_transformers.base.model: No device provided, using cpu


[ChromaStore] Loading embedding model: pritamdeka/S-PubMedBert-MS-MARCO


[2026-04-16 19:29:55,144] INFO — sentence_transformers.base.model: Loading SentenceTransformer model from pritamdeka/S-PubMedBert-MS-MARCO.


[ChromaStore] Opening Chroma store at: ./chroma_store
[ChromaStore] Collection 'pubmed_abstracts' — 56 docs

[ChromaStore] Ready — 56 abstracts in store


In [32]:
# ── PubMed Live Fetch ──────────────────────────────────────────────────────────

PUBMED_MAX_FETCH  = 10     # max abstracts fetched per live query
ENTREZ_EMAIL      = "casm-pipeline@example.com"   # required by NCBI ToS
ENTREZ_TOOL       = "CASM-KnowledgeVerifier"
ENTREZ_SLEEP      = 0.34  # seconds between requests (NCBI 3 req/s limit)


class PubMedFetcher:
    """
    Lightweight NCBI E-utilities client.

    Fetches abstracts for a drug query and upserts them into Chroma.
    Only runs when the Chroma collection does not have enough relevant hits.
    """

    BASE   = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"
    SEARCH = BASE + "/esearch.fcgi"
    FETCH  = BASE + "/efetch.fcgi"
    COMMON = {"tool": ENTREZ_TOOL, "email": ENTREZ_EMAIL, "retmode": "json"}

    def __init__(self, store: ChromaStore):
        """
        Initialise the PubMed client with a target Chroma store.

        Args:
            store : Shared :class:`ChromaStore` used for embedding and upserts.
        """
        self.store = store

    # ── Public API ────────────────────────────────────────────────────────────

    def fetch_and_index(self, drug_name: str, max_results: int = PUBMED_MAX_FETCH) -> int:
        """
        Fetch live PubMed abstracts for a drug and index them in Chroma.

        Args:
            drug_name   : Drug term used to construct the PubMed search query.
            max_results : Maximum number of PMIDs to request from PubMed.

        Returns:
            Number of abstracts upserted into the vector store.
        """
        print(f"  [PubMed] Live fetch for: {drug_name!r}")
        pmids = self._search(drug_name, max_results)
        if not pmids:
            print(f"  [PubMed] No PMIDs found for {drug_name!r}")
            return 0

        abstracts = self._fetch_abstracts(pmids, drug_name)
        n = self.store.upsert(abstracts)
        print(f"  [PubMed] Upserted {n} abstracts for {drug_name!r}")
        return n

    # ── Internal ──────────────────────────────────────────────────────────────

    def _search(self, drug_name: str, max_results: int) -> list[str]:
        """
        Query PubMed ESearch and return PMIDs for a drug-focused query.

        Args:
            drug_name   : Drug term used in MeSH/title-abstract query clauses.
            max_results : Upper bound on PMIDs requested from PubMed.

        Returns:
            A list of PMID strings. Returns an empty list on request failure.
        """
        query = f"{drug_name}[MeSH Terms] OR {drug_name}[Title/Abstract]"
        params = {**self.COMMON, "db": "pubmed", "term": query,
                  "retmax": max_results, "usehistory": "n"}
        try:
            r = requests.get(self.SEARCH, params=params, timeout=10)
            r.raise_for_status()
            return r.json().get("esearchresult", {}).get("idlist", [])
        except Exception as exc:
            print(f"  [PubMed] Search error: {exc}")
            return []

    def _fetch_abstracts(self, pmids: list[str], drug_name: str) -> list[dict]:
        """
        Fetch abstract XML for a PMID batch and parse it into records.

        Args:
            pmids     : PubMed IDs to fetch via EFetch.
            drug_name : Drug tag attached to parsed records for metadata
                        filtering in Chroma.

        Returns:
            Parsed abstract dictionaries ready for :meth:`ChromaStore.upsert`.
            Returns an empty list if EFetch fails.
        """
        ids_str = ",".join(pmids)
        params  = {**self.COMMON, "db": "pubmed", "id": ids_str,
                   "rettype": "abstract", "retmode": "xml"}
        time.sleep(ENTREZ_SLEEP)
        try:
            r = requests.get(self.FETCH, params=params, timeout=30)
            r.raise_for_status()
        except Exception as exc:
            print(f"  [PubMed] Fetch error: {exc}")
            return []

        return self._parse_xml(r.text, drug_name)

    @staticmethod
    def _parse_xml(xml_text: str, drug_name: str) -> list[dict]:
        """
        Parse PubMed XML into lightweight abstract records.

        Args:
            xml_text  : Raw XML response returned by PubMed EFetch.
            drug_name : Drug tag stored with each parsed record.

        Returns:
            A list of dictionaries containing ``pmid``, ``title``,
            ``abstract``, and lower-cased ``drug`` metadata fields.
        """
        articles = []
        # Split on article boundary
        for block in re.split(r"<PubmedArticle>", xml_text)[1:]:
            pmid_m    = re.search(r"<PMID[^>]*>(\d+)</PMID>",        block)
            title_m   = re.search(r"<ArticleTitle>(.*?)</ArticleTitle>", block, re.S)
            abs_m     = re.search(r"<AbstractText[^>]*>(.*?)</AbstractText>", block, re.S)
            if not (pmid_m and abs_m):
                continue
            articles.append({
                "pmid":     pmid_m.group(1),
                "title":    re.sub(r"<[^>]+>", "", title_m.group(1)) if title_m else "",
                "abstract": re.sub(r"<[^>]+>", "", abs_m.group(1)),
                "drug":     drug_name.lower(),
            })
        return articles


pubmed_fetcher = PubMedFetcher(chroma)
print("✅ PubMedFetcher ready")

✅ PubMedFetcher ready


In [33]:
# ── NLI Model ──────────────────────────────────────────────────────────────────

NLI_MODEL_NAME = (
    "lighteternal/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext-finetuned-mnli"
)

# Map model output indices → NLI labels
# HuggingFace MNLI convention:  0=entailment, 1=neutral, 2=contradiction
_NLI_LABEL_MAP = {0: "entailment", 1: "neutral", 2: "contradiction"}

# Internal NLI label → NLIVerdict
_VERDICT_MAP = {
    "entailment":   NLIVerdict.SUPPORTED,
    "neutral":      NLIVerdict.AMBIGUOUS,
    "contradiction": NLIVerdict.CONTRADICTED,
}


class NLIClassifier:
    """
    Thin wrapper around the BiomedNLP-PubMedBERT MNLI model.

    The model is loaded once and kept on the GPU (if available) for the
    duration of the pipeline run.  Each call to :meth:`classify` takes a
    (premise, hypothesis) pair and returns a verdict + confidence score.

    Premise   = PubMed abstract text
    Hypothesis = Claim text to verify
    """

    def __init__(self, model_name: str = NLI_MODEL_NAME):
        """
        Initialise the biomedical MNLI classifier and move it to device.

        Args:
            model_name : HuggingFace model identifier for sequence
                         classification fine-tuned on MNLI.
        """
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"[NLIClassifier] Loading {model_name} on {self.device}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model     = AutoModelForSequenceClassification.from_pretrained(
            model_name
        ).to(self.device)
        self.model.eval()
        print(f"[NLIClassifier] Ready — {sum(p.numel() for p in self.model.parameters()):,} params")

    def classify(self, premise: str, hypothesis: str) -> tuple[NLIVerdict, float]:
        """
        Run NLI inference on a single (premise, hypothesis) pair.

        Args:
            premise    : PubMed abstract text used as evidence.
            hypothesis : Clinical claim text being verified.

        Returns:
            A ``(verdict, confidence)`` tuple where ``verdict`` is an
            :class:`NLIVerdict` and ``confidence`` is the winning softmax
            probability in the range [0, 1].
        """
        inputs = self.tokenizer.encode(
            premise, hypothesis,
            return_tensors="pt",
            truncation=True,
            max_length=512,
        ).to(self.device)

        with torch.no_grad():
            logits = self.model(inputs).logits

        probs      = logits.softmax(dim=-1).cpu().numpy()[0]
        label_idx  = int(np.argmax(probs))
        label_str  = _NLI_LABEL_MAP[label_idx]
        confidence = float(probs[label_idx])
        verdict    = _VERDICT_MAP[label_str]
        return verdict, confidence

    def classify_batch(
        self, premise: str, hypotheses: list[str]
    ) -> list[tuple[NLIVerdict, float]]:
        """
        Classify one premise against multiple hypotheses.

        Current implementation is sequential and delegates each pair to
        :meth:`classify`; true tensor batching is intentionally left as a
        future optimisation.

        Args:
            premise    : Shared evidence text to compare against each claim.
            hypotheses : List of claim strings to evaluate.

        Returns:
            List of ``(verdict, confidence)`` outputs in input order.
        """
        return [self.classify(premise, h) for h in hypotheses]


nli = NLIClassifier()
print("\n✅ NLIClassifier ready")

[NLIClassifier] Loading lighteternal/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext-finetuned-mnli on cpu
[NLIClassifier] Ready — 109,484,547 params

✅ NLIClassifier ready


In [34]:
# ── openFDA Client ─────────────────────────────────────────────────────────────

FDA_BASE    = "https://api.fda.gov/drug/event.json"
FDA_TIMEOUT = 10  # seconds


class OpenFDAClient:
    """
    Query the openFDA /drug/event.json endpoint for adverse event counts.

    Two counts are returned per drug:
      - ``total``   : All adverse event reports mentioning the drug.
      - ``serious`` : Reports where ``serious=1`` (hospitalisation, death, etc.).

    The client is deliberately stateless — each call is independent.
    Rate limit: 240 requests/minute without an API key (sufficient here).
    """

    @staticmethod
    def _count(drug_name: str, extra_filter: str = "") -> int:
        """
        Execute a single openFDA count query.

        Args:
            drug_name    : Drug name to query in the medicinal product field.
            extra_filter : Optional additional openFDA search clause appended
                           with ``AND`` (e.g. ``"serious:1"``).

        Returns:
            Integer total returned by openFDA. Returns ``0`` for missing drugs
            (HTTP 404) or request failures.
        """
        search = f'patient.drug.medicinalproduct:"{drug_name}"'
        if extra_filter:
            search += f" AND {extra_filter}"
        params = {"search": search, "limit": 1}
        try:
            r = requests.get(FDA_BASE, params=params, timeout=FDA_TIMEOUT)
            if r.status_code == 404:
                return 0          # drug not found — not an error
            r.raise_for_status()
            return r.json().get("meta", {}).get("results", {}).get("total", 0)
        except Exception as exc:
            print(f"  [openFDA] Query error for {drug_name!r}: {exc}")
            return 0

    def query(self, drug_name: str) -> tuple[int, int]:
        """
        Return ``(total_events, serious_events)`` for a drug name.

        Args:
            drug_name : Drug name as it appears in the claim
                        (e.g. ``"Ibuprofen"``).

        Returns:
            A ``(total, serious)`` tuple from openFDA count queries.
        """
        if not drug_name:
            return 0, 0
        total   = self._count(drug_name)
        serious = self._count(drug_name, extra_filter="serious:1")
        return total, serious


fda_client = OpenFDAClient()
print("✅ OpenFDAClient ready")

✅ OpenFDAClient ready


In [35]:
import numpy as np
from collections import Counter

# ── NLI Aggregation (Updated for Decisive Logic) ───────────────────────────────

def aggregate_verdicts(
    verdicts:    list, # Assuming NLIVerdict enum
    confidences: list[float],
) -> tuple:
    """
    Combine per-abstract NLI results into a single claim-level verdict.

    Rules
    -----
    1. Filter out AMBIGUOUS (noise) if decisive evidence exists.
    2. The majority DECISIVE label wins.
    3. In a tie between CONTRADICTED and SUPPORTED, CONTRADICTED wins
       (conservative medical safety bias).
    4. The returned confidence is the weighted mean of confidences for the
       winning-verdict abstracts.
    """
    if not verdicts:
        return NLIVerdict.AMBIGUOUS, 0.0

    # 1. Filter out AMBIGUOUS to see if we have real evidence
    decisive_indices = [i for i, v in enumerate(verdicts) if v != NLIVerdict.AMBIGUOUS]

    # 2. If all papers were AMBIGUOUS (no decisive evidence found)
    if not decisive_indices:
        return NLIVerdict.AMBIGUOUS, float(np.mean(confidences))

    # 3. We have decisive evidence! Count the votes.
    decisive_verdicts = [verdicts[i] for i in decisive_indices]
    counts = Counter(decisive_verdicts)

    sup_votes = counts.get(NLIVerdict.SUPPORTED, 0)
    con_votes = counts.get(NLIVerdict.CONTRADICTED, 0)

    # 4. Determine winner based on decisive votes (ties go to CONTRADICTED)
    if con_votes >= sup_votes:
        winner = NLIVerdict.CONTRADICTED
    else:
        winner = NLIVerdict.SUPPORTED

    # 5. Calculate mean confidence of abstracts that voted for the winner
    winning_confs = [c for v, c in zip(verdicts, confidences) if v == winner]
    mean_conf = float(np.mean(winning_confs)) if winning_confs else 0.0

    return winner, mean_conf


# ── Self-Tests ─────────────────────────────────────────────────────────────────

_test_cases = [
    # Case 1: The "Sitagliptin" scenario from your logs.
    # 2 Supported, 3 Ambiguous -> Should now return SUPPORTED
    ([NLIVerdict.AMBIGUOUS]*3 + [NLIVerdict.SUPPORTED]*2,
     [0.96, 0.55, 0.99, 0.95, 0.88], NLIVerdict.SUPPORTED),

    # Case 2: Standard clear support
    ([NLIVerdict.SUPPORTED]*3 + [NLIVerdict.AMBIGUOUS]*2,
     [0.9]*5, NLIVerdict.SUPPORTED),

    # Case 3: Conflicting evidence tie -> CONTRADICTED wins for safety
    ([NLIVerdict.CONTRADICTED]*2 + [NLIVerdict.SUPPORTED]*2 + [NLIVerdict.AMBIGUOUS],
     [0.8]*5, NLIVerdict.CONTRADICTED),

    # Case 4: The "Metformin" scenario. All noise -> AMBIGUOUS
    ([NLIVerdict.AMBIGUOUS]*5,
     [0.6]*5, NLIVerdict.AMBIGUOUS),
]

for i, (vs, cs, expected) in enumerate(_test_cases):
    got, _ = aggregate_verdicts(vs, cs)
    assert got == expected, f"Test {i+1} failed! Expected {expected}, got {got}"

print("✅ aggregate_verdicts — all self-tests passed (including Sitagliptin fix!)")

✅ aggregate_verdicts — all self-tests passed (including Sitagliptin fix!)


In [36]:
# ── Knowledge Verifier ─────────────────────────────────────────────────────────

class KnowledgeVerifier:
    """
    CASM Stage 2 — Knowledge Verifier.

    Accepts a list of Claim objects (from the ClaimExtractor) and returns
    a parallel list of EvidenceResult objects.

    For claims with ``requires_verification=True``:
      1. Encode claim → Chroma semantic search (top-K abstracts).
      2. If Chroma has too few results, live-fetch from PubMed and upsert.
      3. Run NLI on each abstract in a thread pool (NLI + FDA overlap).
      4. Aggregate per-abstract NLI verdicts into one claim-level verdict.
      5. Query openFDA for adverse-event counts (runs in parallel with NLI).
      6. Return EvidenceResult.

    For claims with ``requires_verification=False``:
      Returns a skipped EvidenceResult immediately.

    Parameters
    ----------
    chroma_store   : Shared ChromaStore instance.
    nli_classifier : Shared NLIClassifier instance.
    fda_client     : Shared OpenFDAClient instance.
    pubmed_fetcher : Shared PubMedFetcher instance.
    top_k          : Number of abstracts to retrieve per claim.
    min_hits       : Minimum Chroma hits before triggering live fetch.
    """

    def __init__(
        self,
        chroma_store:   ChromaStore,
        nli_classifier: NLIClassifier,
        fda_client:     OpenFDAClient,
        pubmed_fetcher: PubMedFetcher,
        top_k:          int = TOP_K,
        min_hits:       int = MIN_HITS,
    ):
        """
        Initialise the verifier with shared pipeline components.

        Args:
            chroma_store   : Persistent vector store used for semantic retrieval.
            nli_classifier : Biomedical NLI model wrapper for claim validation.
            fda_client     : Client for openFDA adverse-event count queries.
            pubmed_fetcher : Live PubMed fetch helper used for cache warm-up.
            top_k          : Number of abstracts to retrieve per claim.
            min_hits       : Minimum acceptable Chroma hits before triggering
                             a live PubMed backfill.
        """
        self.chroma    = chroma_store
        self.nli       = nli_classifier
        self.fda       = fda_client
        self.fetcher   = pubmed_fetcher
        self.top_k     = top_k
        self.min_hits  = min_hits

    # ── Public API ────────────────────────────────────────────────────────────

    def verify_all(self, claims: list) -> list[EvidenceResult]:
        """
        Verify a list of Claim objects and return one EvidenceResult per claim.

        Args:
            claims : Output of ClaimExtractor.extract() — list of Claim dataclass
                     objects. Plain dicts (from extract_as_dict) are also accepted.

        Returns:
            List of :class:`EvidenceResult` objects in the same order as
            the input claims.
        """
        results = []
        for claim in claims:
            # Support both dataclass and dict representations
            if isinstance(claim, dict):
                claim_id   = claim["claim_id"]
                claim_text = claim["claim_text"]
                drug_names = claim.get("drug_names", [])
                req_verif  = claim.get("requires_verification", False)
            else:
                claim_id   = claim.claim_id
                claim_text = claim.claim_text
                drug_names = claim.drug_names
                req_verif  = claim.requires_verification

            print(f"\n[Verifier] {claim_id} — requires_verification={req_verif}")
            print(f"  claim: {claim_text[:100]}")

            if not req_verif:
                results.append(EvidenceResult(
                    claim_id=claim_id, claim_text=claim_text, skipped=True
                ))
                print("  → Skipped (no verification required)")
                continue

            result = self._verify_single(claim_id, claim_text, drug_names)
            results.append(result)

        return results

    # ── Internal ──────────────────────────────────────────────────────────────

    def _verify_single(
        self,
        claim_id:   str,
        claim_text: str,
        drug_names: list[str],
    ) -> EvidenceResult:
        """
        Run the full retrieval + verification workflow for one claim.

        Steps:
          1. Query Chroma for top-k semantic neighbours.
          2. Optionally trigger live PubMed fetch when hits are insufficient.
          3. Run NLI over retrieved abstracts in parallel with openFDA queries.
          4. Aggregate per-abstract NLI verdicts into one claim-level verdict.

        Args:
            claim_id   : Stable claim identifier (e.g. ``"claim_0"``).
            claim_text : Raw claim text to verify.
            drug_names : Candidate drug names extracted from the claim.

        Returns:
            A populated :class:`EvidenceResult`, including error metadata when
            an exception occurs.
        """
        primary_drug = drug_names[0] if drug_names else None

        try:
            # ── Step 1+2: Chroma semantic search ──────────────────────────────
            hits = self.chroma.query(claim_text, self.top_k, drug_filter=primary_drug)
            print(f"  [Step 2] Chroma returned {len(hits)} hit(s)")

            # ── Step 3: Live PubMed fetch if not enough hits ──────────────────
            if len(hits) < self.min_hits and primary_drug:
                print(f"  [Step 3] Too few hits ({len(hits)} < {self.min_hits}) — fetching from PubMed")
                self.fetcher.fetch_and_index(primary_drug)
                hits = self.chroma.query(claim_text, self.top_k, drug_filter=primary_drug)
                print(f"  [Step 3] After fetch: {len(hits)} hit(s)")

            if not hits:
                return EvidenceResult(
                    claim_id=claim_id, claim_text=claim_text,
                    verdict=NLIVerdict.AMBIGUOUS, confidence=0.0,
                    error="No PubMed abstracts found",
                )

            # ── Steps 4+5: NLI and openFDA in parallel ────────────────────────
            with concurrent.futures.ThreadPoolExecutor(max_workers=2) as pool:
                nli_future = pool.submit(self._run_nli,   claim_text, hits)
                fda_future = pool.submit(self._run_fda,   primary_drug)

                verdicts, scores = nli_future.result()
                fda_total, fda_serious = fda_future.result()

            # ── Step 5: Aggregate ─────────────────────────────────────────────
            agg_verdict, agg_conf = aggregate_verdicts(verdicts, scores)

            print(f"  [Step 5] Verdict: {agg_verdict.value}  Confidence: {agg_conf:.3f}")
            print(f"  [Step 5] FDA total={fda_total}  serious={fda_serious}")

            return EvidenceResult(
                claim_id     = claim_id,
                claim_text   = claim_text,
                verdict      = agg_verdict,
                confidence   = agg_conf,
                pubmed_hits  = hits,
                fda_count    = fda_total,
                fda_serious  = fda_serious,
                nli_scores   = scores,
                nli_verdicts = [v.value for v in verdicts],
            )

        except Exception as exc:
            import traceback
            print(f"  [Verifier] ERROR: {exc}")
            traceback.print_exc()
            return EvidenceResult(
                claim_id=claim_id, claim_text=claim_text,
                error=str(exc),
            )

    def _run_nli(
        self,
        claim_text: str,
        hits: list[SearchResult],
    ) -> tuple[list[NLIVerdict], list[float]]:
        """
        Run NLI inference for one claim against all retrieved abstracts.

        Args:
            claim_text : Hypothesis text being validated.
            hits       : Retrieved PubMed abstracts used as NLI premises.

        Returns:
            Tuple of parallel lists ``(verdicts, scores)`` in hit order.
        """
        verdicts, scores = [], []
        for hit in hits:
            verdict, conf = self.nli.classify(
                premise    = hit.abstract,
                hypothesis = claim_text,
            )
            verdicts.append(verdict)
            scores.append(conf)
            print(f"    [NLI] pmid={hit.pmid}  {verdict.value}  conf={conf:.3f}")
        return verdicts, scores

    def _run_fda(self, drug_name: Optional[str]) -> tuple[int, int]:
        """
        Query openFDA event counts for the primary claim drug.

        Args:
            drug_name : Drug name extracted from the claim, if available.

        Returns:
            Tuple ``(total_events, serious_events)``. Returns ``(0, 0)`` when
            no drug name is provided.
        """
        if not drug_name:
            return 0, 0
        total, serious = self.fda.query(drug_name)
        return total, serious


# Instantiate the verifier
verifier = KnowledgeVerifier(
    chroma_store   = chroma,
    nli_classifier = nli,
    fda_client     = fda_client,
    pubmed_fetcher = pubmed_fetcher,
)
print("\n✅ KnowledgeVerifier ready")


✅ KnowledgeVerifier ready


In [37]:
# # ── Test: Verify Claims from ClaimExtractor Output ─────────────────────────────
# #
# # Three scenarios taken from the ClaimExtractor test cells above.
# # Each call uses the dict output from extract_as_dict() so this cell can
# # run standalone without re-instantiating the ClaimExtractor.
# #
# # NOTE: The first run will trigger live PubMed fetches for drugs not yet in the
# # Chroma store. Subsequent runs will hit the local vector cache instead.
#
# import json
#
# # ── Scenario A: Corneal haze / FML ────────────────────────────────────────────
# scenario_a = [
#     {
#         "claim_id": "claim_0",
#         "claim_text": "prescribe Fluorometholone (FML) 0.1% eye drops four times daily for 4 weeks",
#         "claim_type": "DOSAGE_CLAIM",
#         "drug_names": ["Fluorometholone"],
#         "dosages": ["0.1%"],
#         "conditions": [],
#         "requires_verification": True,
#         "confidence": 0.0,
#     },
#     {
#         "claim_id": "claim_1",
#         "claim_text": "Consider Oral Vitamin C (1000mg/day) to support corneal healing",
#         "claim_type": "DOSAGE_CLAIM",
#         "drug_names": ["Vitamin C"],
#         "dosages": ["1000mg"],
#         "conditions": [],
#         "requires_verification": True,
#         "confidence": 0.0,
#     },
# ]
#
# print("=" * 70)
# print("SCENARIO A — Corneal haze / FML + Vitamin C")
# print("=" * 70)
# results_a = verifier.verify_all(scenario_a)
#
# # ── Scenario B: UTI treatment ─────────────────────────────────────────────────
# scenario_b = [
#     {
#         "claim_id": "claim_0",
#         "claim_text": "prescribe Trimethoprim 200mg twice daily for 7 days",
#         "claim_type": "DOSAGE_CLAIM",
#         "drug_names": ["Trimethoprim"],
#         "dosages": ["200mg"],
#         "conditions": ["UTI"],
#         "requires_verification": True,
#         "confidence": 0.0,
#     },
#     {
#         "claim_id": "claim_1",
#         "claim_text": "switch to Nitrofurantoin 100mg four times daily for 5 days",
#         "claim_type": "DOSAGE_CLAIM",
#         "drug_names": ["Nitrofurantoin"],
#         "dosages": ["100mg"],
#         "conditions": [],
#         "requires_verification": True,
#         "confidence": 0.0,
#     },
#     {
#         "claim_id": "claim_2",
#         "claim_text": "Monitor renal function with creatinine levels weekly",
#         "claim_type": "PROCEDURAL_CLAIM",
#         "drug_names": [],
#         "dosages": [],
#         "conditions": [],
#         "requires_verification": False,
#         "confidence": 0.0,
#     },
# ]
#
# print("\n" + "=" * 70)
# print("SCENARIO B — UTI / Trimethoprim + Nitrofurantoin")
# print("=" * 70)
# results_b = verifier.verify_all(scenario_b)
#
# # ── Scenario C: T2DM / CKD — high-risk claim ─────────────────────────────────
# scenario_c = [
#     {
#         "claim_id": "claim_0",
#         "claim_text": "avoid Metformin 500mg if eGFR drops below 30 ml/min due to risk of lactic acidosis",
#         "claim_type": "DRUG_SAFETY_CLAIM",
#         "drug_names": ["Metformin"],
#         "dosages": ["500mg", "30 ml/min"],
#         "conditions": ["CKD", "T2DM"],
#         "requires_verification": True,
#         "confidence": 0.0,
#     },
#     {
#         "claim_id": "claim_1",
#         "claim_text": "Consider switching to Sitagliptin 50mg once daily, which is renally dosed and safer in this population",
#         "claim_type": "DOSAGE_CLAIM",
#         "drug_names": ["Sitagliptin"],
#         "dosages": ["50mg"],
#         "conditions": ["CKD"],
#         "requires_verification": True,
#         "confidence": 0.0,
#     },
# ]
#
# print("\n" + "=" * 70)
# print("SCENARIO C — T2DM/CKD / Metformin safety + Sitagliptin")
# print("=" * 70)
# results_c = verifier.verify_all(scenario_c)
#
# print("\n✅ All scenarios complete")

In [38]:
# # ── Results Display ────────────────────────────────────────────────────────────
#
# def print_evidence_table(results: list[EvidenceResult], title: str = "") -> None:
#     """
#     Pretty-print a list of :class:`EvidenceResult` objects.
#
#     Displays claim-level verdicts, confidence, openFDA counts, and per-abstract
#     NLI outcomes in a readable console table for quick manual inspection.
#
#     Args:
#         results : Verification outputs to display, typically returned by
#                   :meth:`KnowledgeVerifier.verify_all`.
#         title   : Optional section header printed above the table.
#
#     Returns:
#         ``None``. This function is side-effect only (stdout printing).
#     """
#     VERDICT_ICON = {
#         "SUPPORTED":    "✅",
#         "CONTRADICTED": "❌",
#         "AMBIGUOUS":    "⚠️ ",
#     }
#
#     if title:
#         print(f"\n{'═' * 72}")
#         print(f"  {title}")
#         print(f"{'═' * 72}")
#
#     for r in results:
#         icon = "⏭️ " if r.skipped else VERDICT_ICON.get(r.verdict.value, "?")
#         print(f"\n[{r.claim_id}] {icon} {r.verdict.value if not r.skipped else 'SKIPPED'}")
#         print(f"  Claim      : {r.claim_text[:90]}")
#         if r.skipped:
#             print("  (no verification required for this claim type)")
#             continue
#         if r.error:
#             print(f"  ERROR      : {r.error}")
#             continue
#         print(f"  Confidence : {r.confidence:.3f}")
#         print(f"  FDA events : total={r.fda_count:,}  serious={r.fda_serious:,}")
#         print(f"  Abstracts  : {len(r.pubmed_hits)} used")
#         for i, (pmid_hit, verdict, score) in enumerate(
#             zip(r.pubmed_hits, r.nli_verdicts, r.nli_scores)
#         ):
#             v_icon = VERDICT_ICON.get(verdict, "?")
#             print(f"    [{i+1}] pmid={pmid_hit.pmid}  {v_icon} {verdict}  conf={score:.3f}  dist={pmid_hit.distance:.3f}")
#             if pmid_hit.title:
#                 print(f"         title: {pmid_hit.title[:75]}")
#
# print_evidence_table(results_a, "Scenario A — Corneal haze")
# print_evidence_table(results_b, "Scenario B — UTI treatment")
# print_evidence_table(results_c, "Scenario C — T2DM / CKD safety")
#
# # ── JSON export ───────────────────────────────────────────────────────────────
# all_results = {
#     "scenario_a": [r.to_dict() for r in results_a],
#     "scenario_b": [r.to_dict() for r in results_b],
#     "scenario_c": [r.to_dict() for r in results_c],
# }
# with open("knowledge_verifier_results.json", "w") as f:
#     json.dump(all_results, f, indent=2)
#
# print("\n\n✅ Results saved → knowledge_verifier_results.json")

In [39]:
# ── FastAPI Server for CASM ────────────────────────────────────────────────────────
#
# This cell sets up a FastAPI server so external users can:
#   1. Extract claims via POST /extract
#   2. Verify claims via POST /verify
#   3. Run full pipeline via POST /analyze
#   4. Run concise pipeline via POST /analyze-verdict
#
# Usage:
#   1. Run this cell to start the server
#   2. Visit http://localhost:8000/docs for interactive API documentation
#   3. Submit requests via curl, Python requests, or Swagger UI

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import List, Optional
import uvicorn
import logging

# ── Logging Setup ──────────────────────────────────────────────────────────────────

logging.basicConfig(
    level=logging.INFO,
    format="[%(asctime)s] %(levelname)s — %(name)s: %(message)s"
)
logger = logging.getLogger("CASM-API")

# ── Pydantic Request/Response Models ───────────────────────────────────────────────

class ExtractRequest(BaseModel):
    """Request to extract claims from raw LLM response."""
    llm_response: str = Field(..., description="Raw clinical text from LLM", min_length=1)


class ExtractResponse(BaseModel):
    """Response containing extracted claims."""
    success: bool = True
    claims: List[dict] = Field(..., description="List of extracted claims")
    claim_count: int = Field(..., description="Number of claims extracted")
    error: Optional[str] = None


class VerifyRequest(BaseModel):
    """Request to verify a list of claims."""
    claims: List[dict] = Field(..., description="Claims to verify", min_items=1)


class VerifyResponse(BaseModel):
    """Response containing verification evidence."""
    success: bool = True
    evidence: List[dict] = Field(..., description="Verification results for each claim")
    verified_count: int = Field(..., description="Number of claims verified")
    error: Optional[str] = None


class AnalyzeRequest(BaseModel):
    """Request for full end-to-end pipeline."""
    llm_response: str = Field(..., description="Raw clinical text from LLM", min_length=1)


class AnalyzeResponse(BaseModel):
    """Full end-to-end pipeline response."""
    success: bool = True
    claims: List[dict] = Field(..., description="Extracted claims")
    evidence: List[dict] = Field(..., description="Verification results")
    claim_count: int
    verified_count: int
    error: Optional[str] = None


class ConciseVerdict(BaseModel):
    """A stripped-down model containing only the claim, its verdict, and sources."""
    claim_text: str = Field(..., description="The extracted clinical claim")
    verdict: str = Field(..., description="The verification verdict (e.g., Supported, Refuted)")
    sources: list = Field(default_factory=list, description="Paper sources or PubMed abstracts")


class VerdictOnlyResponse(BaseModel):
    """Response containing only verdicts and sources."""
    success: bool = True
    results: List[ConciseVerdict] = Field(..., description="List of concise verdicts")
    error: Optional[str] = None


class HealthResponse(BaseModel):
    """Health check response."""
    status: str
    version: str = "1.0.0"
    models_ready: bool
    extractors: dict = Field(default_factory=dict)


# ── FastAPI App Setup ──────────────────────────────────────────────────────────────

app = FastAPI(
    title="CASM — Clinical Automated Safety Monitor",
    description="Two-stage NLP pipeline for extracting and verifying clinical claims",
    version="1.0.0",
    docs_url="/docs",
    redoc_url="/redoc",
    openapi_url="/openapi.json",
)

# Global instances (shared across requests)
_extractor: Optional['ClaimExtractor'] = None
_verifier: Optional['KnowledgeVerifier'] = None

logger.info("FastAPI app initialized")

# ── Health & Diagnostics ──────────────────────────────────────────────────────────

@app.get("/health", response_model=HealthResponse)
async def health_check():
    """
    Health check endpoint.

    Returns status of models and API readiness.
    """
    return HealthResponse(
        status="healthy" if _extractor and _verifier else "degraded",
        models_ready=bool(_extractor and _verifier),
        extractors={
            "extractor_loaded": _extractor is not None,
            "verifier_loaded": _verifier is not None,
        }
    )


@app.get("/")
async def root():
    """Root endpoint with API information."""
    return {
        "service": "CASM — Clinical Automated Safety Monitor",
        "version": "1.0.0",
        "documentation": "http://localhost:8000/docs",
        "health": "http://localhost:8000/health",
        "endpoints": {
            "extract": "POST /extract — Extract claims only",
            "verify": "POST /verify — Verify existing claims",
            "analyze": "POST /analyze — Full pipeline (extract + verify)",
            "analyze-verdict": "POST /analyze-verdict — Concise pipeline (verdicts + sources only)"
        }
    }

# ── Stage 1: Claim Extraction ──────────────────────────────────────────────────────

@app.post("/extract", response_model=ExtractResponse)
async def extract_claims(request: ExtractRequest) -> ExtractResponse:
    """
    Extract claims from raw LLM clinical response.

    **Input:**
    - `llm_response`: Raw text from an LLM describing a clinical scenario

    **Output:**
    - List of structured claims with drug names, dosages, conditions, etc.
    """
    try:
        if _extractor is None:
            raise HTTPException(status_code=503, detail="Extractor not loaded")

        logger.info(f"Extracting from {len(request.llm_response)} char response")
        claims = _extractor.extract_as_dict(request.llm_response)

        logger.info(f"Extracted {len(claims)} claim(s)")
        return ExtractResponse(
            success=True,
            claims=claims,
            claim_count=len(claims),
        )
    except Exception as e:
        logger.exception("Extract failed")
        raise HTTPException(status_code=500, detail=f"Extraction failed: {str(e)}")


# ── Stage 2: Claim Verification ────────────────────────────────────────────────────

@app.post("/verify", response_model=VerifyResponse)
async def verify_claims(request: VerifyRequest) -> VerifyResponse:
    """
    Verify a list of claims using PubMed + NLI + openFDA.

    **Input:**
    - `claims`: List of claim dictionaries (from /extract endpoint or manual creation)

    **Output:**
    - Verification evidence for each claim (verdict, confidence, PubMed abstracts, FDA counts)
    """
    try:
        if _verifier is None:
            raise HTTPException(status_code=503, detail="Verifier not loaded")

        logger.info(f"Verifying {len(request.claims)} claim(s)")
        results = _verifier.verify_all(request.claims)

        evidence = [r.to_dict() for r in results]
        logger.info(f"Verification complete — {len(evidence)} results")

        return VerifyResponse(
            success=True,
            evidence=evidence,
            verified_count=len(evidence),
        )
    except Exception as e:
        logger.exception("Verify failed")
        raise HTTPException(status_code=500, detail=f"Verification failed: {str(e)}")


# ── Full Pipeline: Extract + Verify ────────────────────────────────────────────────

@app.post("/analyze", response_model=AnalyzeResponse)
async def full_pipeline(request: AnalyzeRequest) -> AnalyzeResponse:
    """
    End-to-end pipeline: Extract claims → Verify against evidence → Return results.

    This is the primary endpoint for users who have raw LLM output.

    **Input:**
    - `llm_response`: Raw clinical text from LLM

    **Output:**
    - Extracted claims + verification evidence
    """
    try:
        if _extractor is None or _verifier is None:
            raise HTTPException(status_code=503, detail="Models not loaded")

        logger.info("=" * 70)
        logger.info("FULL PIPELINE START")
        logger.info("=" * 70)

        # ── Stage 1: Extract ──
        logger.info("[Stage 1] Extracting claims...")
        claims = _extractor.extract_as_dict(request.llm_response)
        logger.info(f"[Stage 1] ✅ Extracted {len(claims)} claim(s)")

        # ── Stage 2: Verify ──
        logger.info("[Stage 2] Verifying claims...")
        results = _verifier.verify_all(claims)
        evidence = [r.to_dict() for r in results]
        logger.info(f"[Stage 2] ✅ Verified {len(evidence)} claim(s)")

        logger.info("=" * 70)
        logger.info("FULL PIPELINE COMPLETE")
        logger.info("=" * 70)

        return AnalyzeResponse(
            success=True,
            claims=claims,
            evidence=evidence,
            claim_count=len(claims),
            verified_count=len(evidence),
        )
    except Exception as e:
        logger.exception("Full pipeline failed")
        raise HTTPException(status_code=500, detail=f"Pipeline failed: {str(e)}")


# ── Concise Pipeline: Verdict + Sources Only ───────────────────────────────────────

@app.post("/analyze-verdict", response_model=VerdictOnlyResponse)
async def analyze_verdict_only(request: AnalyzeRequest) -> VerdictOnlyResponse:
    """
    Streamlined pipeline: Runs extraction and verification, but returns ONLY
    the extracted claim text, its verdict, and the paper sources.

    **Input:**
    - `llm_response`: Raw clinical text from LLM
    """
    try:
        if _extractor is None or _verifier is None:
            raise HTTPException(status_code=503, detail="Models not loaded")

        logger.info("Running streamlined Verdict-Only pipeline...")

        # 1. Extract
        claims = _extractor.extract_as_dict(request.llm_response)

        # 2. Verify
        results = _verifier.verify_all(claims)

        # 3. Filter down to just Verdict and Sources
        concise_results = []
        for claim, ev in zip(claims, results):
            ev_dict = ev.to_dict()

            # Safely grab paper sources (looks for common CASM dictionary keys)
            sources = ev_dict.get("pubmed_abstracts") or ev_dict.get("sources") or ev_dict.get("evidence") or []

            concise_results.append(
                ConciseVerdict(
                    claim_text=claim.get("claim_text", "Unknown Claim"),
                    verdict=ev_dict.get("verdict", "Unknown Verdict"),
                    sources=sources
                )
            )

        logger.info(f"Verdict-Only pipeline complete — processed {len(concise_results)} claim(s)")
        return VerdictOnlyResponse(success=True, results=concise_results)

    except Exception as e:
        logger.exception("Verdict-only pipeline failed")
        raise HTTPException(status_code=500, detail=f"Pipeline failed: {str(e)}")


# ── Error Handlers ─────────────────────────────────────────────────────────────────

@app.exception_handler(HTTPException)
async def http_exception_handler(request, exc):
    """Custom HTTP exception handler."""
    return {
        "success": False,
        "error": exc.detail,
        "status_code": exc.status_code,
    }


# ── Server Initialization ──────────────────────────────────────────────────────────

def init_models():
    """Initialize extractor and verifier models."""
    global _extractor, _verifier

    logger.info("Initializing models...")
    try:
        logger.info("Loading ClaimExtractor...")
        _extractor = testdict  # Make sure 'testdict' is defined in a previous cell!
        logger.info("✅ ClaimExtractor loaded")

        logger.info("Loading KnowledgeVerifier...")
        _verifier = verifier  # Make sure 'verifier' is defined in a previous cell!
        logger.info("✅ KnowledgeVerifier loaded")

        logger.info("=" * 70)
        logger.info("✅ ALL MODELS INITIALIZED")
        logger.info("=" * 70)
    except Exception as e:
        logger.error(f"❌ Failed to initialize models: {e}")
        raise


# ── Startup & Shutdown ─────────────────────────────────────────────────────────────

@app.on_event("startup")
async def startup():
    """Initialize models when server starts."""
    logger.info("FastAPI server starting...")
    init_models()
    logger.info("Server ready to accept requests")


@app.on_event("shutdown")
async def shutdown():
    """Cleanup when server shuts down."""
    logger.info("FastAPI server shutting down...")
    logger.info("Goodbye!")


# ── Run Server (Jupyter/IPython Compatible) ────────────────────────────────────────

if __name__ == "__main__":
    logger.info("Starting CASM API server...")
    logger.info("📖 API docs: http://localhost:8000/docs")
    logger.info("🔍 ReDoc: http://localhost:8000/redoc")

    # Create the Uvicorn configuration and server instance
    config = uvicorn.Config(app, port=8000, reload=False)
    server = uvicorn.Server(config)

    # Await the server directly instead of using uvicorn.run()
    await server.serve()

C:\Users\basim\AppData\Local\Temp\ipykernel_20396\1726099260.py:45: PydanticDeprecatedSince20: `min_items` is deprecated and will be removed, use `min_length` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  claims: List[dict] = Field(..., description="Claims to verify", min_items=1)
[2026-04-16 19:30:02,894] INFO — CASM-API: FastAPI app initialized
C:\Users\basim\AppData\Local\Temp\ipykernel_20396\1726099260.py:344: DeprecationWarning: 
        on_event is deprecated, use lifespan event handlers instead.

        Read more about it in the
        [FastAPI docs for Lifespan Events](https://fastapi.tiangolo.com/advanced/events/).
        
  @app.on_event("startup")
C:\Users\basim\AppData\Local\Temp\ipykernel_20396\1726099260.py:352: DeprecationWarning: 
        on_event is deprecated, use lifespan event handlers instead.

        Read more about it in the
        [FastAPI docs for Lifespan Events

INFO:     127.0.0.1:54717 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:54717 - "GET /openapi.json HTTP/1.1" 200 OK


[2026-04-16 19:31:16,029] INFO — CASM-API: Running streamlined Verdict-Only pipeline...
C:\Users\basim\PycharmProjects\Clinical-AI-Safety-Monitor\.venv\Lib\site-packages\thinc\shims\pytorch.py:114: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(self._mixed_precision):


[ClaimExtractor] 5 claim(s) extracted.

[Verifier] claim_0 — requires_verification=False
  claim: In a patient with Type 2 Diabetes Mellitus and Chronic Kidney Disease, continuation of Metformin is 
  → Skipped (no verification required)

[Verifier] claim_1 — requires_verification=True
  claim: Metformin should be discontinued if eGFR falls below 30 mL/min/1.73 m² due to the increased risk of 


Batches: 100%|██████████| 1/1 [00:00<00:00, 11.99it/s]

...

  [Step 2] Chroma returned 5 hit(s)


    [NLI] pmid=41987548  AMBIGUOUS  conf=0.927
    [NLI] pmid=41987878  AMBIGUOUS  conf=0.977
    [NLI] pmid=41985330  AMBIGUOUS  conf=0.608
    [NLI] pmid=41948417  AMBIGUOUS  conf=0.931


[2026-04-16 19:31:24,563] INFO — CASM-API: Verdict-Only pipeline complete — processed 5 claim(s)


    [NLI] pmid=41914136  AMBIGUOUS  conf=0.691
  [Step 5] Verdict: AMBIGUOUS  Confidence: 0.827
  [Step 5] FDA total=0  serious=0

[Verifier] claim_4 — requires_verification=False
  claim: HbA1c should be monitored every 3 months, and renal function should be reassessed periodically to gu
  → Skipped (no verification required)
INFO:     127.0.0.1:56949 - "POST /analyze-verdict HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
[2026-04-16 19:33:13,978] INFO — CASM-API: FastAPI server shutting down...
[2026-04-16 19:33:13,978] INFO — CASM-API: Goodbye!
INFO:     Application shutdown complete.
INFO:     Finished server process [20396]


In [40]:
# Run this in a new cell
test_claim = [{"claim_id": "test", "claim_text": "Sitagliptin for diabetes", "requires_verification": True}]
test_result = _verifier.verify_all(test_claim)[0].to_dict()

print("🔍 AVAILABLE KEYS IN EVIDENCE DICTIONARY:")
for key in test_result.keys():
    print(f" - {key}")


[Verifier] test — requires_verification=True
  claim: Sitagliptin for diabetes


Batches: 100%|██████████| 1/1 [00:00<00:00, 12.66it/s]

  [Step 2] Chroma returned 5 hit(s)
    [NLI] pmid=41918146  SUPPORTED  conf=0.935


    [NLI] pmid=41987548  AMBIGUOUS  conf=0.728
    [NLI] pmid=41962900  SUPPORTED  conf=0.774
    [NLI] pmid=41985330  CONTRADICTED  conf=0.945
    [NLI] pmid=41948417  CONTRADICTED  conf=0.747
  [Step 5] Verdict: CONTRADICTED  Confidence: 0.846
  [Step 5] FDA total=0  serious=0
🔍 AVAILABLE KEYS IN EVIDENCE DICTIONARY:
 - claim_id
 - claim_text
 - verdict
 - confidence
 - fda_count
 - fda_serious
 - nli_scores
 - nli_verdicts
 - pubmed_hits
 - skipped
 - error
